### **Environment Setup & Package Installation**

In [ ]:
%cd /content
!rm -rf drone_detection
from google.colab import userdata
gh_token = (userdata.get("GH_TOKEN") or "").strip()
assert gh_token, "GH_TOKEN secret is empty or missing"
!git clone --depth 1 https://{gh_token}@github.com/Hannysia/drone_detection.git
!pip install -r drone_detection/requirements-colab.txt -q
!pip install -e drone_detection/ --no-deps -q
%cd /content/drone_detection

In [ ]:
import logging
from pathlib import Path
import kagglehub
from IPython.display import display
from ultralytics import YOLO, settings

from src.config import OUTPUT_DIR, WORKING_DIR
from src.processing.diagnostics import compare_models_shortcut_learning
from src.processing.evaluation import (
    analyze_model_errors,
    calibrate_confidence_thresholds,
    download_yolo_run_artifacts,
    evaluate_runs_on_test,
    run_dataset_predictions_db,
)
from src.utils.visualization import (
    plot_error_analysis_dashboard,
    plot_error_gallery,
    plot_yolo_runs_comparison,
)

logging.basicConfig(level=logging.INFO, format="%(message)s")
settings.update({"wandb": False})

DATASET_ROOT = Path(kagglehub.dataset_download("hannusiaknysh/drone-detection-curated-v1"))
EVAL_DIR = OUTPUT_DIR / "evaluation_v1"
IMGSZ = 1280
RUN_NAMES = ("run_positives_only", "run_with_negatives")

artifacts = download_yolo_run_artifacts(RUN_NAMES, cache_dir=WORKING_DIR / "hf_artifacts")

### **1. Training Dynamics Comparison (`Positives Only` vs `With Negatives`)**

In [ ]:
comparison_df = plot_yolo_runs_comparison(
    {
        "Positives Only": artifacts["run_positives_only"]["results_csv"],
        "With Negatives": artifacts["run_with_negatives"]["results_csv"],
    },
    save_path=EVAL_DIR / "yolo_runs_comparison.png",
    show=True,
)
display(comparison_df)

### **2. Shortcut Learning & Paired EigenCAM Diagnostics (`val` Split)**

In [ ]:
shortcut_report = compare_models_shortcut_learning(
    {
        "Positives Only": artifacts["run_positives_only"]["weights"],
        "With Negatives": artifacts["run_with_negatives"]["weights"],
    },
    DATASET_ROOT,
    EVAL_DIR / "shortcut_comparison",
    split="val",
    conf_thresh=0.25,
    n_negative_pairs=5,
    n_positive_pairs=5,
    imgsz=IMGSZ,
)
display(shortcut_report["summary"])

### **3. Confidence Threshold Calibration (`val` Split — `F0.5` & `F1`)**

In [ ]:
raw_database = run_dataset_predictions_db(
    YOLO(str(artifacts["run_with_negatives"]["weights"])),
    DATASET_ROOT,
    imgsz=IMGSZ,
    collection_conf=0.01,
    split="val",
)
calibration = calibrate_confidence_thresholds(
    raw_database,
    save_path=EVAL_DIR / "threshold_calibration.png",
    show=True,
)
display(calibration["summary"])

### **4. Final Locked Test Split Evaluation (Both Models)**

In [ ]:
test_results_df = evaluate_runs_on_test(
    {
        run_name: artifacts[run_name]["weights"]
        for run_name in RUN_NAMES
    },
    DATASET_ROOT,
    calibrated_run="run_with_negatives",
    calibrated_conf=calibration["opt_threshold_f05"],
    imgsz=IMGSZ,
    batch=16,
)
display(test_results_df)

### **5. Residual Error Analysis (Locked Test Split @ `F0.5`)**

test_database = run_dataset_predictions_db(
    YOLO(str(artifacts["run_with_negatives"]["weights"])),
    DATASET_ROOT,
    imgsz=IMGSZ,
    collection_conf=0.01,
    split="test",
)
error_report = analyze_model_errors(
    test_database,
    conf_threshold=calibration["opt_threshold_f05"],
    top_k=6,
)
plot_error_analysis_dashboard(
    error_report,
    save_path=EVAL_DIR / "error_analysis_dashboard.png",
    show=True,
)
for key in ("by_size", "by_position", "by_background", "by_camera", "by_shape", "error_types"):
    print(f"\n--- {key} ---")
    display(error_report[key])

plot_error_gallery(
    error_report["worst_fps"],
    kind="fp",
    title="Top Confident False Positives (Test Split @ F0.5)",
    save_path=EVAL_DIR / "worst_false_positives.png",
    show=True,
)
plot_error_gallery(
    error_report["worst_fns"],
    kind="fn",
    title="Largest Missed Targets / False Negatives (Test Split @ F0.5)",
    save_path=EVAL_DIR / "worst_false_negatives.png",
    show=True,
)